# Day 4 · 从原始数据到数据仓库

> **核心任务**：用 PySpark 把杂乱的 CSV/TSV 加工成 ODS / DWD 两层 Parquet 数据仓库

> **学习关键词**：数仓分层、维度建模、Parquet 列存、Hive 风格分区

## 数据仓库分层 —— 数据工程师的核心知识

**为什么要分层**：原始数据脏、乱、散落多处。如果分析师每次都直接读原始 CSV,会出现"100 个分析师 100 套清洗逻辑",数据混乱。分层数仓的意义：**让脏活只做一次,中间结果可复用**。

经典四层（阿里巴巴大数据之路提出的标准）：

| 层级 | 含义 | 内容 | 我们今天做的 |
|---|---|---|---|
| **ODS** | 原始数据层 | 几乎不动原始数据,只做格式转换、字符编码统一 | ✅ |
| **DWD** | 明细数据层 | 清洗 + 维度关联 + 标准化,形成"干净的明细表" | ✅ |
| DWS | 汇总数据层 | 按主题/维度做轻度聚合（用户日活、内容日热） | Day 5 做 |
| ADS | 应用数据层 | 面向具体看板的指标表 | Day 5 做 |

## 今天的重点概念

1. **Parquet 比 CSV 优秀在哪**：列式存储 → 只读需要的列 → 查询快 10 倍 + 文件小 5 倍
2. **Hive 风格分区**：按某个维度（通常是时间）把数据切成多个目录,查询时只扫需要的目录
3. **维度建模 vs 范式建模**：维度建模为分析而生,**牺牲存储换取查询速度**

---
## 0. 启动 Spark Session

**注意**：每个 Notebook 启动时都要重新建 SparkSession。这里我们配置得更完整一些。

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pathlib import Path

spark = SparkSession.builder \
    .appName("ContentDataPlatform-Day4") \
    .master("local[*]") \
    .config("spark.driver.memory", "3g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# 项目路径
PROJECT_ROOT = Path('/home/rein/projects/content-data-platform')
RAW = PROJECT_ROOT / 'data' / 'raw'
WAREHOUSE = PROJECT_ROOT / 'data' / 'warehouse'
ODS = WAREHOUSE / 'ods'
DWD = WAREHOUSE / 'dwd'

ODS.mkdir(parents=True, exist_ok=True)
DWD.mkdir(parents=True, exist_ok=True)

print(f'Spark version: {spark.version}')
print(f'Warehouse: {WAREHOUSE}')
print('✅ 准备就绪')

---
# Part 1 · ODS 层建设

## 1.1 ODS 层的设计原则

**ODS = Operational Data Store = 操作型数据存储**

原则：**几乎不改原始数据,只做格式转换**。具体允许做的事：
- 把 CSV/TSV/JSON 等格式统一成 Parquet
- 修正字段类型（比如把字符串日期转成 DATE 类型）
- 统一空值表示（`\N`、空串、NULL 统一为 NULL）
- 加一列 `ingest_time`（入仓时间,方便后续追溯）

**不允许做的事**：
- 关联其他表（那是 DWD 层做的）
- 聚合（那是 DWS 层做的）
- 删除数据（哪怕看起来脏,也要保留,让 DWD 层决定怎么处理）

## 1.2 为什么转 Parquet

In [ ]:
# 先看一下原始 ratings.csv 有多大
import os
csv_path = RAW / 'movielens' / 'ml-25m' / 'ratings.csv'
csv_size_mb = os.path.getsize(csv_path) / 1024 / 1024
print(f'原始 ratings.csv 大小：{csv_size_mb:.1f} MB')

650MB 的 CSV,稍后转成 Parquet,我们看会变成多大。

## 1.3 ODS 任务 1：MovieLens ratings 转 Parquet

In [ ]:
print('📥 读取 ratings.csv...')
ratings_raw = spark.read.option("header", "true").csv(str(RAW / 'movielens' / 'ml-25m' / 'ratings.csv'))

# 看一下原始 schema(Spark 默认所有列都是 string)
ratings_raw.printSchema()

**注意**：Spark 默认推断不出类型,所有字段都是 string。**ODS 层我们要修正成正确的类型** —— 这是 ODS 层允许做的事之一。

In [ ]:
from pyspark.sql.functions import col, current_timestamp

# 修正类型 + 加 ingest_time
ratings_ods = ratings_raw \
    .withColumn("userId", col("userId").cast("int")) \
    .withColumn("movieId", col("movieId").cast("int")) \
    .withColumn("rating", col("rating").cast("double")) \
    .withColumn("timestamp", col("timestamp").cast("long")) \
    .withColumn("ingest_time", current_timestamp())

ratings_ods.printSchema()
ratings_ods.show(5)

In [ ]:
# 写入 Parquet —— 这一步要等 1-2 分钟,正在处理 2500 万行
import time
start = time.time()

ratings_ods.write \
    .mode("overwrite") \
    .parquet(str(ODS / 'ml_ratings'))

elapsed = time.time() - start
print(f'✅ 写完 ods.ml_ratings,耗时 {elapsed:.1f} 秒')

In [ ]:
# 见证奇迹:Parquet 比 CSV 小多少?
def get_dir_size(path):
    total = 0
    for p in Path(path).rglob('*'):
        if p.is_file():
            total += p.stat().st_size
    return total / 1024 / 1024

parquet_size = get_dir_size(ODS / 'ml_ratings')
print(f'原始 CSV：{csv_size_mb:.1f} MB')
print(f'Parquet：{parquet_size:.1f} MB')
print(f'压缩比：{csv_size_mb / parquet_size:.1f}x')

**Parquet 通常比 CSV 小 5-10 倍**,因为：
- 列式存储（同列数据类型一致,压缩效率高）
- 默认 Snappy 压缩
- 自带 schema（不需要每行重复列名）

## 1.4 验证：Parquet 查询比 CSV 快多少

In [ ]:
# 对比同一个查询的速度
import time

# CSV 查询
start = time.time()
csv_count = spark.read.option("header", "true").csv(str(csv_path)).count()
csv_elapsed = time.time() - start

# Parquet 查询
start = time.time()
parquet_count = spark.read.parquet(str(ODS / 'ml_ratings')).count()
parquet_elapsed = time.time() - start

print(f'CSV count:     {csv_elapsed:.2f} 秒')
print(f'Parquet count: {parquet_elapsed:.2f} 秒')
print(f'加速：{csv_elapsed / parquet_elapsed:.1f}x')
print(f'(都是 {csv_count:,} 行)')

## 1.5 ODS 任务 2：批量转换其他数据源

把剩下的 4 个原始数据源也转成 Parquet。**写成函数,工程化思维**。

In [ ]:
def ods_load_csv(spark, source_path, target_path, type_overrides=None, **read_options):
    """
    通用 ODS 加载函数：从源文件读取 -> 修正类型 -> 加 ingest_time -> 写 Parquet
    
    type_overrides: dict, 列名 -> 目标类型
    read_options: 传给 spark.read 的额外选项(比如 delimiter='\t')
    """
    reader = spark.read
    for key, value in read_options.items():
        reader = reader.option(key, value)
    
    df = reader.csv(str(source_path))
    
    if type_overrides:
        for col_name, target_type in type_overrides.items():
            if col_name in df.columns:
                df = df.withColumn(col_name, col(col_name).cast(target_type))
    
    df = df.withColumn("ingest_time", current_timestamp())
    df.write.mode("overwrite").parquet(str(target_path))
    
    count = spark.read.parquet(str(target_path)).count()
    print(f'✅ {target_path.name}: {count:,} 行')
    return count

In [ ]:
# 转 ml_movies
ods_load_csv(
    spark,
    source_path=RAW / 'movielens' / 'ml-25m' / 'movies.csv',
    target_path=ODS / 'ml_movies',
    type_overrides={'movieId': 'int'},
    header='true'
)

# 转 ml_links
ods_load_csv(
    spark,
    source_path=RAW / 'movielens' / 'ml-25m' / 'links.csv',
    target_path=ODS / 'ml_links',
    type_overrides={'movieId': 'int', 'imdbId': 'int', 'tmdbId': 'int'},
    header='true'
)

In [ ]:
# 转 IMDb 数据 —— 这两个用 TSV 分隔符 + \N 表示 null
ods_load_csv(
    spark,
    source_path=RAW / 'imdb' / 'title.basics.tsv.gz',
    target_path=ODS / 'imdb_titles',
    type_overrides={'isAdult': 'int', 'startYear': 'int', 'endYear': 'int', 'runtimeMinutes': 'int'},
    header='true',
    delimiter='\t',
    nullValue='\\N'
)

ods_load_csv(
    spark,
    source_path=RAW / 'imdb' / 'title.ratings.tsv.gz',
    target_path=ODS / 'imdb_ratings',
    type_overrides={'averageRating': 'double', 'numVotes': 'int'},
    header='true',
    delimiter='\t',
    nullValue='\\N'
)

print('\n🎉 ODS 层全部就绪')

In [ ]:
# 看一下 ODS 层的全貌
import os
for table_dir in sorted(ODS.iterdir()):
    if table_dir.is_dir():
        size_mb = get_dir_size(table_dir)
        print(f'  {table_dir.name:20s} {size_mb:7.1f} MB')

print(f'\nODS 层总大小：{get_dir_size(ODS):.1f} MB')

---
# Part 2 · DWD 层建设

## 2.1 DWD 层的设计原则

**DWD = Data Warehouse Detail = 明细数据层**

做的事：
- **清洗**：去重、补默认值、过滤无效记录
- **维度退化**：把多个 ODS 表 JOIN 起来,形成"事实表 + 维度表"结构
- **标准化字段命名**：让所有下游看到统一的字段名（如 `event_time` 而不是 `timestamp`）
- **建立基础维度**：dim_xxx 表是"对某类实体的全方位描述"

## 2.2 维度建模简介

**两种基本表**：

- **维度表 dim_xxx**：描述"实体"是什么（一部电影、一个用户、一个时间点）
  - 例：`dim_movie` 包含 movie_id、title、year、genres、director、avg_rating 等
  
- **事实表 fact_xxx**：记录"事件"发生了什么（一次评分、一次播放、一次购买）
  - 例：`fact_user_rating` 包含 user_id、movie_id、rating、event_time

**星型模型**：一张事实表 + 多张维度表围绕在四周,JOIN 时事实表是中心。

## 2.3 我们今天要建的两张 DWD 表

1. **dim_movie**（电影维度表）：合并 ml_movies + ml_links + imdb_titles + imdb_ratings,形成一张大维表
2. **fact_user_rating**（评分事实表）：清洗后的 ml_ratings,按年分区

## 2.4 注册 ODS 表为临时视图

**Spark SQL 习惯做法**：把要用的 Parquet 注册成 SQL 视图,后续用 SQL 操作。

In [ ]:
spark.read.parquet(str(ODS / 'ml_ratings')).createOrReplaceTempView('ods_ml_ratings')
spark.read.parquet(str(ODS / 'ml_movies')).createOrReplaceTempView('ods_ml_movies')
spark.read.parquet(str(ODS / 'ml_links')).createOrReplaceTempView('ods_ml_links')
spark.read.parquet(str(ODS / 'imdb_titles')).createOrReplaceTempView('ods_imdb_titles')
spark.read.parquet(str(ODS / 'imdb_ratings')).createOrReplaceTempView('ods_imdb_ratings')

spark.sql("SHOW TABLES").show()

## 2.5 构建 dim_movie

**业务定义**：dim_movie 是"对一部电影的全方位描述",包含：
- MovieLens 自己的 ID 和标题、类型
- IMDb 的标题、年份、时长、IMDb 评分
- TMDb ID（后续拉海报用）
- 衍生字段：年代分桶、是否高分、热度等级

**关键决策**：以 MovieLens 的 movieId 为主键。原因：用户评分事实表用的是 movieId,主键统一才好 JOIN。

In [ ]:
dim_movie = spark.sql("""
    WITH ml_with_imdb AS (
        -- 把 ml_links.imdbId 转成 IMDb 的 tconst 格式
        SELECT 
            m.movieId,
            m.title AS ml_title,
            m.genres AS ml_genres,
            l.imdbId,
            l.tmdbId,
            CONCAT('tt', LPAD(CAST(l.imdbId AS STRING), 7, '0')) AS tconst
        FROM ods_ml_movies m
        LEFT JOIN ods_ml_links l ON m.movieId = l.movieId
    )
    SELECT 
        -- 主键
        mli.movieId,
        
        -- 基础信息(优先用 IMDb 数据,降级用 MovieLens)
        COALESCE(t.primaryTitle, mli.ml_title) AS title,
        t.originalTitle,
        t.startYear AS release_year,
        t.runtimeMinutes AS runtime_minutes,
        COALESCE(t.genres, mli.ml_genres) AS genres,
        
        -- 外部 ID
        mli.tconst AS imdb_id,
        mli.tmdbId AS tmdb_id,
        
        -- IMDb 评分数据
        r.averageRating AS imdb_rating,
        r.numVotes AS imdb_votes,
        
        -- 衍生字段:年代分桶
        CASE 
            WHEN t.startYear < 1950 THEN '黄金时代'
            WHEN t.startYear < 1980 THEN '经典时代'
            WHEN t.startYear < 2000 THEN '现代时代'
            WHEN t.startYear < 2010 THEN '新世纪初'
            WHEN t.startYear >= 2010 THEN '当代'
            ELSE '未知'
        END AS era,
        
        -- 衍生字段:口碑等级
        CASE 
            WHEN r.averageRating >= 8.0 AND r.numVotes >= 100000 THEN '神作'
            WHEN r.averageRating >= 7.0 AND r.numVotes >= 50000 THEN '佳作'
            WHEN r.averageRating >= 6.0 AND r.numVotes >= 10000 THEN '良作'
            WHEN r.averageRating < 6.0 AND r.numVotes >= 10000 THEN '平庸'
            WHEN r.averageRating >= 7.0 AND r.numVotes < 10000 THEN '小众佳作'
            ELSE '待考察'
        END AS quality_tier,
        
        -- 元数据
        current_timestamp() AS dwd_etl_time
        
    FROM ml_with_imdb mli
    LEFT JOIN ods_imdb_titles t ON mli.tconst = t.tconst
    LEFT JOIN ods_imdb_ratings r ON mli.tconst = r.tconst
    WHERE t.titleType IN ('movie', 'tvMovie') OR t.titleType IS NULL
""")

dim_movie.printSchema()

In [ ]:
# 看一眼数据
dim_movie.show(5, truncate=False)
print(f'\n总行数：{dim_movie.count():,}')

In [ ]:
# 写入 DWD 层
dim_movie.write.mode("overwrite").parquet(str(DWD / 'dim_movie'))
print(f'✅ dwd.dim_movie 写入完成')
print(f'   文件大小：{get_dir_size(DWD / "dim_movie"):.1f} MB')

## 2.6 数据质量验证

**真实工作中,每张 DWD 表写完都要做 DQ 检查**。这是数据工程师的职业素养。

In [ ]:
spark.read.parquet(str(DWD / 'dim_movie')).createOrReplaceTempView('dwd_dim_movie')

# DQ 1：主键唯一性
dq1 = spark.sql("""
    SELECT COUNT(*) AS total, COUNT(DISTINCT movieId) AS unique_ids
    FROM dwd_dim_movie
""").collect()[0]
print(f'DQ1 主键唯一性：total={dq1["total"]:,}, unique={dq1["unique_ids"]:,}, '
      f'{"✅ PASS" if dq1["total"] == dq1["unique_ids"] else "❌ FAIL"}')

# DQ 2：质量等级分布
print('\nDQ2 质量等级分布：')
spark.sql("""
    SELECT quality_tier, COUNT(*) AS cnt
    FROM dwd_dim_movie
    GROUP BY quality_tier
    ORDER BY cnt DESC
""").show()

# DQ 3：年代分布
print('DQ3 年代分布：')
spark.sql("""
    SELECT era, COUNT(*) AS cnt
    FROM dwd_dim_movie
    GROUP BY era
    ORDER BY 
        CASE era WHEN '黄金时代' THEN 1 WHEN '经典时代' THEN 2 WHEN '现代时代' THEN 3
                 WHEN '新世纪初' THEN 4 WHEN '当代' THEN 5 ELSE 6 END
""").show()

---
## 2.7 构建 fact_user_rating —— 重点中的重点

这是项目里**第一张分区表**,直接对应面试时"Hive 分区怎么用"的问题。

**为什么要分区**：
- 2500 万行数据,如果当一个大文件存,每次查询都要扫全表
- 按年份分区后,查询"2015 年的数据"只需要扫 `year=2015/` 目录
- 这叫做 **分区裁剪(Partition Pruning)**,是 Hive/Spark 的核心优化

**Hive 风格分区的物理形态**：
```
data/warehouse/dwd/fact_user_rating/
├── year=1996/part-00000.parquet
├── year=1997/part-00000.parquet
├── year=1998/part-00000.parquet
└── ...
```

每个分区就是一个文件夹,目录名 `year=1996` 就是分区键的值。

In [ ]:
fact_user_rating = spark.sql("""
    SELECT 
        r.userId AS user_id,
        r.movieId AS movie_id,
        r.rating,
        
        -- 时间维度展开
        from_unixtime(r.timestamp) AS event_time,
        YEAR(from_unixtime(r.timestamp)) AS year,
        MONTH(from_unixtime(r.timestamp)) AS month,
        DAY(from_unixtime(r.timestamp)) AS day,
        DAYOFWEEK(from_unixtime(r.timestamp)) AS day_of_week,
        
        -- 评分分类(方便 DWS 层聚合)
        CASE 
            WHEN r.rating >= 4.0 THEN 'positive'
            WHEN r.rating >= 3.0 THEN 'neutral'
            ELSE 'negative'
        END AS sentiment,
        
        -- 是否极端评分(用于异常用户检测)
        CASE WHEN r.rating IN (5.0, 0.5) THEN 1 ELSE 0 END AS is_extreme,
        
        current_timestamp() AS dwd_etl_time
    FROM ods_ml_ratings r
    WHERE r.rating IS NOT NULL
      AND r.userId IS NOT NULL
      AND r.movieId IS NOT NULL
""")

fact_user_rating.printSchema()
fact_user_rating.show(5)

In [ ]:
# 关键写入语句:partitionBy('year') 让 Spark 按 year 字段值切目录
import time
start = time.time()

fact_user_rating.write \
    .mode("overwrite") \
    .partitionBy("year") \
    .parquet(str(DWD / 'fact_user_rating'))

print(f'✅ 写入完成,耗时 {time.time()-start:.1f} 秒')

In [ ]:
# 看分区目录结构
import subprocess
result = subprocess.run(
    ['ls', str(DWD / 'fact_user_rating')],
    capture_output=True, text=True
)
print('fact_user_rating 目录结构：')
for line in sorted(result.stdout.split('\n'))[:30]:
    print(f'  {line}')

**漂亮 ✨** —— 你刚刚创建了项目里第一张 Hive 风格分区表。这个目录结构跟生产环境里 Hive/Spark 数仓**一模一样**。

## 2.8 验证分区裁剪的效果

**对比两个查询**：
- 查询 A：跨年统计 → 需要扫所有分区
- 查询 B：单年统计 → 只扫一个分区

In [ ]:
spark.read.parquet(str(DWD / 'fact_user_rating')).createOrReplaceTempView('dwd_fact_user_rating')

# 查询 A：扫所有分区
start = time.time()
result_a = spark.sql("""
    SELECT year, COUNT(*) as cnt
    FROM dwd_fact_user_rating
    GROUP BY year ORDER BY year
""").collect()
elapsed_a = time.time() - start
print(f'查询 A(扫所有分区): {elapsed_a:.2f} 秒')

# 查询 B：只扫一个分区(WHERE year=...)
start = time.time()
result_b = spark.sql("""
    SELECT COUNT(*) as cnt, AVG(rating) as avg_rating
    FROM dwd_fact_user_rating
    WHERE year = 2015
""").collect()
elapsed_b = time.time() - start
print(f'查询 B(单分区): {elapsed_b:.2f} 秒')

print(f'\n分区裁剪加速比：{elapsed_a / elapsed_b:.1f}x')

**面试问"为什么 Hive 表要分区"的标准答案**：
- 减少 IO（只读需要的分区）
- 并行处理（不同分区可以并行扫）
- 分区维度通常是时间,符合大多数业务查询模式

你现在亲眼见到了分区裁剪的效果,以后讲这个不再是背的。

---
# Part 3 · 验证：在数据仓库上重跑 Day 3 的查询

**关键时刻** —— 用你 Day 3 在 DuckDB 上写过的 SQL,直接在你新建的 DWD 数据仓库上跑一遍,看结果是不是一致。

In [ ]:
# 经典题:每个类型的 Top 3 电影
spark.sql("""
    WITH movie_genre_exploded AS (
        SELECT 
            title,
            release_year,
            imdb_rating,
            imdb_votes,
            EXPLODE(SPLIT(genres, ',')) AS genre
        FROM dwd_dim_movie
        WHERE imdb_rating IS NOT NULL
          AND imdb_votes >= 100000
          AND genres IS NOT NULL
    ),
    ranked AS (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY genre ORDER BY imdb_rating DESC, imdb_votes DESC) AS rn
        FROM movie_genre_exploded
    )
    SELECT genre, title, release_year, imdb_rating, imdb_votes
    FROM ranked WHERE rn <= 3
    ORDER BY genre, rn
    LIMIT 30
""").show(30, truncate=False)

**注意一个细节**：DuckDB 里 `UNNEST(STRING_SPLIT(...))`,这里 Spark 里是 `EXPLODE(SPLIT(...))`。

**这是 SQL 方言差异** —— 90% 一样,10% 函数名不同。Hive、Spark、ClickHouse、PostgreSQL、DuckDB 都有自己的方言。这部分你不用死记,**遇到 Spark 报错 "function not found" 时去查文档替换就行**。

## 3.2 跨事实表 + 维度表的查询

**展示星型模型的威力** —— 一个查询同时关联 fact 表和 dim 表,这是数仓的核心查询模式。

In [ ]:
# 业务问题:2018 年用户给"神作"打的平均分,跟给"平庸"打的平均分有什么差异?
spark.sql("""
    SELECT 
        m.quality_tier,
        COUNT(*) AS rating_count,
        ROUND(AVG(f.rating), 2) AS user_avg_rating,
        ROUND(AVG(m.imdb_rating), 2) AS imdb_avg_rating
    FROM dwd_fact_user_rating f
    JOIN dwd_dim_movie m ON f.movie_id = m.movieId
    WHERE f.year = 2018
      AND m.quality_tier != '待考察'
    GROUP BY m.quality_tier
    ORDER BY user_avg_rating DESC
""").show()

**这个查询的妙处**：
- WHERE year=2018 触发了**分区裁剪**,只扫一个目录
- JOIN dwd_dim_movie 用上了我们刚建的**维度表**
- 同一个查询既有 fact 又有 dim,**这就是星型模型的查询模式**

你刚刚体验了一次**真正的数据仓库查询**。

---
# Part 4 · 总览

看看你今天建好的数据仓库整体形态。

In [ ]:
print('=' * 60)
print('📊 你的数据仓库')
print('=' * 60)

print('\n📂 ODS 层(原始数据,只改格式):')
for d in sorted(ODS.iterdir()):
    if d.is_dir():
        print(f'   {d.name:25s} {get_dir_size(d):7.1f} MB')

print('\n📂 DWD 层(清洗与维度建模):')
for d in sorted(DWD.iterdir()):
    if d.is_dir():
        size = get_dir_size(d)
        # 看是不是分区表
        partition_count = len([p for p in d.iterdir() if p.is_dir() and '=' in p.name])
        marker = f'(✨ {partition_count} 个分区)' if partition_count > 0 else ''
        print(f'   {d.name:25s} {size:7.1f} MB  {marker}')

print('\n📊 总大小：')
print(f'   ODS：{get_dir_size(ODS):.1f} MB')
print(f'   DWD：{get_dir_size(DWD):.1f} MB')
print(f'   总计：{get_dir_size(WAREHOUSE):.1f} MB')
print('=' * 60)

## 关掉 Spark

In [ ]:
spark.stop()
print('Spark 已关闭')

---
# 🎉 Day 4 完成

## 你今天获得的能力

1. **数仓分层思维** —— 不再认为"数据加工"是混乱的一坨,而是有清晰的 ODS / DWD 分层逻辑
2. **维度建模** —— 你建了人生第一张 `dim_movie` 维度表 + `fact_user_rating` 事实表
3. **Hive 风格分区** —— 你亲眼见证了分区裁剪的速度提升,**这是面试高频题**
4. **Parquet 列存** —— 知道为什么大数据领域都用 Parquet 而不是 CSV/JSON
5. **数据质量校验(DQ)** —— 学到了"写完表要做主键唯一性、行数对账"等工程素养

## 简历上可以写的话术

> 基于 PySpark + Parquet 构建 ODS / DWD 两层离线数仓,完成多源异构数据（IMDb + MovieLens）的清洗与整合,设计 `dim_movie` 维度表与 `fact_user_rating` 事实表,采用 Hive 风格按年分区优化查询性能,实现分区裁剪后查询提速 N 倍。

## 明天 Day 5

继续往上建 **DWS（汇总层）+ ADS（应用层）**,把今天的明细数据加工成可以直接喂给前端看板的指标表。